# 01 — Schema, types & missingness (team 07)

**Owner:** Kingston · **Branch:** `team-07-kingston`

**Questions to answer**
1. **Shape & types** — rows, columns, and what each column *actually* means. Use Snowflake's
   column descriptions (ask Patrick for access details) — don't guess at a real dataset's schema.
   Build a data dictionary: column · dtype · meaning · example of what a value represents (no real values).
2. What is **one row**? (one route per month? one airport pair per day?) This decides everything else.
3. **Missingness** — which columns have gaps, how much (%), and is it random or tied to something
   (a route, an airport, a date range)? Check this with a groupby, don't eyeball it.
4. Did 00 drop many **duplicate rows** or convert anything it shouldn't have? Flag it to Patrick.

**Write the "why", not just the plot.** Every plot or number needs a sentence after it:
*"…therefore we should ___ when we model."* Those sentences go in the table at the bottom.

> **NDA reminder.** This is real American Airlines data. Never commit it, never paste rows or
> screenshots of it anywhere outside the team, and **clear outputs before every commit**
> (Kernel → Restart & Clear Output) so no data ends up in the notebook file.

In [ ]:
%run ./00_load_and_preprocess.ipynb
# gives you: pax, airports, TARGET, REPO_ROOT, RESULTS_DIR

## 1. Shape, types, data dictionary

In [ ]:
print("pax:", pax.shape, "| airports:", airports.shape)

# "Unnamed: 0" is just the CSV's saved index (0, 1, 2, ...), not a real column
print("Unnamed: 0 is a row counter:", (pax["Unnamed: 0"].values == range(len(pax))).all())

meaning = {
    "Unnamed: 0":  "leftover pandas index from the export, no meaning",
    "DEP_YEAR":    "year of departure",
    "DEP_MONTH":   "month of departure (1-12)",
    "ORG_AIRPRT":  "origin airport code (3 letters)",
    "DES_AIRPRT":  "destination airport code (3 letters)",
    "OD_PAX":      "origin-destination passengers in that month (TARGET)",
    "OD_REV_USD":  "revenue from those passengers, in USD",
    "SF_LOAD_TMS": "Snowflake load timestamp, same for every row",
}
example = {
    "Unnamed: 0":  "a row number",
    "DEP_YEAR":    "a year, 2019-2026",
    "DEP_MONTH":   "a month number",
    "ORG_AIRPRT":  "an airport code like a hub",
    "DES_AIRPRT":  "an airport code",
    "OD_PAX":      "a passenger count for a route-month",
    "OD_REV_USD":  "a dollar total for a route-month",
    "SF_LOAD_TMS": "when the table was loaded",
}
data_dict = pd.DataFrame({
    "dtype": pax.dtypes.astype(str),
    "n_unique": pax.nunique(),
    "meaning": pd.Series(meaning),
    "example": pd.Series(example),
})
data_dict

Answer: The main table (pax) has about 1.5 million rows and 8 columns. Two of the columns are useless. Unnamed: 0 is just a row number left over from saving the file, and SF_LOAD_TMS has the same value in every row. The other 6 columns are the year, the month, the origin airport, the destination airport, passengers (OD_PAX, our target), and revenue (OD_REV_USD). The types look correct. Airport codes are text, and the year, month, passengers and revenue are numbers.

What this means for the model: We should drop Unnamed: 0 and SF_LOAD_TMS. We should also combine year and month into one date column so we can split the data by time.

Note: the meanings in the table are based on the column names. We still need to check them against the Snowflake descriptions (ask Patrick).

## 2. What is one row?

In [ ]:
key = ["DEP_YEAR", "DEP_MONTH", "ORG_AIRPRT", "DES_AIRPRT"]
print("duplicate rows on key:", pax.duplicated(subset=key).sum())
print("rows where origin == destination:", (pax["ORG_AIRPRT"] == pax["DES_AIRPRT"]).sum())

# is A->B a different row from B->A?
routes = pax[["ORG_AIRPRT", "DES_AIRPRT"]].drop_duplicates()
reverse = routes.rename(columns={"ORG_AIRPRT": "DES_AIRPRT", "DES_AIRPRT": "ORG_AIRPRT"})
print("routes:", len(routes), "| routes whose reverse also exists:", len(routes.merge(reverse)))

Answer: One row is one route (origin to destination) for one month. Year + month + origin + destination never repeats, so those four columns identify every row. A to B and B to A count as two different routes.

What this means for the model: Each route is its own monthly time series. Any lag or rolling-average features should be calculated separately for each route.

## 3. Missingness

In [ ]:
# 1) classic missing values
print("% NaN per column (pax):")
print((pax.isna().mean() * 100).round(2).to_string())
print("\n% NaN per column (airports), only columns with gaps:")
air_na = airports.isna().mean() * 100
print(air_na[air_na > 0].round(2).to_string())

# 2) "hidden" missingness: route-months that simply aren't in the table
n_months = pax[key[:2]].drop_duplicates().shape[0]
rows_per_route = pax.groupby(["ORG_AIRPRT", "DES_AIRPRT"]).size()
print(f"\nmonths covered: {n_months} | routes: {len(rows_per_route)}")
print(f"route-months present: {len(pax) / (len(rows_per_route) * n_months):.1%} of the full grid")
print(f"routes with every month: {(rows_per_route == n_months).mean():.1%}")

# 3) is it tied to time? rows per year (2026 is a partial year)
print("\nrows per year / months in that year:")
per_year = pax.groupby("DEP_YEAR").agg(rows=("OD_PAX", "size"), months=("DEP_MONTH", "nunique"))
per_year["rows_per_month"] = (per_year["rows"] / per_year["months"]).round(0)
print(per_year.to_string())

# 4) other sanity checks
print("\nOD_PAX <= 0:", (pax["OD_PAX"] <= 0).sum(), "| OD_REV_USD <= 0:", (pax["OD_REV_USD"] <= 0).sum())
codes = set(airports["AIRPRT_CD"])
print("pax airports missing from AIRPORT_INFO:",
      len((set(pax["ORG_AIRPRT"]) | set(pax["DES_AIRPRT"])) - codes))

# 5) are the gaps tied to an airport or to small routes?
missing = (n_months - rows_per_route).rename("missing_months").reset_index()
by_airport = (missing.groupby("ORG_AIRPRT")["missing_months"].sum()
              / (missing.groupby("ORG_AIRPRT").size() * n_months) * 100)
print("\n% of route-months missing, per origin airport:")
print(by_airport.describe().round(2).to_string())

avg_pax = pax.groupby(["ORG_AIRPRT", "DES_AIRPRT"])["OD_PAX"].mean().values
missing["has_gap"] = missing["missing_months"] > 0
missing["avg_pax"] = avg_pax
print("\nmedian avg passengers: routes with gaps vs. routes with no gaps")
print(missing.groupby("has_gap")["avg_pax"].median().round(0).to_string())

# 6) which months in 2020 lost rows?
print("\nrows per month in 2020:")
print(pax[pax["DEP_YEAR"] == 2020].groupby("DEP_MONTH").size().to_string())


Answer: No column in pax has any missing values (0% everywhere). The real problem is that some route-months are missing completely. About 1.6% of all route-months aren't in the table, and about 35% of routes are missing at least one month. These gaps are not random. Most of the missing rows are in 2020, especially April through June, which lines up with COVID. Routes with gaps also carry far fewer passengers than routes with no gaps, about 10 times fewer on average, so small routes probably just had months with no passengers recorded. The gaps are spread across airports and aren't concentrated in just a few.

In the airport table, STATE_PROVNC_CD is about 52% missing, probably because airports outside the US don't have a state. Every airport in pax is also in the airport table.

What this means for the model: We need to ask Patrick whether a missing month means 0 passengers or just no data, before we fill anything in. We should also treat 2020 as an unusual year.

Q4, did 00 drop duplicates or convert anything? No. 00 dropped 0 duplicate rows and didn't convert any columns, so it didn't change the data. (Flag this to Patrick.)

## Findings → modelling consequence (Kingston)

This table is what the team's feature set (05) is built from. Every row needs a **consequence**,
not just an observation — "*therefore we should ___ when we model.*"

| # | Observation (with the number) | Modelling consequence |
|---|---|---|
| 1 | One row = one route in one month. 0 duplicates, about 17,000 routes, 91 months (Jan 2019 to Jul 2026) | Treat each route as its own time series and split train/test by date, not randomly |
| 2 | `Unnamed: 0` is just a row number and `SF_LOAD_TMS` has only 1 value | Drop both columns |
| 3 | No NaNs, but about 1.6% of route-months are missing, mostly in 2020 and on small routes | Ask Patrick if missing means 0 passengers before filling it in. Treat 2020 as an unusual year |
| 4 | Airport table: state is about 52% missing, and every `pax` airport has a match | We can join the airport table safely. Use country instead of state |